# Issue #355 — low-N match selector

Executed decision notebook.

In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT=Path.cwd()
OUT=ROOT/'results/low_n_match_selector'
decision=json.loads((OUT/'decision.json').read_text())
decision

{'changes_210_or_218': False,
 'contract': 'BJ-UNIFIED-LOWN-MATCH-SELECTOR-001/v1',
 'development_n_with_positive_hit_lift': 2,
 'improved_oof_calibration_vs_s0': True,
 'issue': 355,
 'no_catastrophic_transfer_reversal': False,
 'production_or_holdout_authorized': False,
 'real_selection_changes': True,
 'selected_family': 'S2_LOGISTIC',
 'transfer_reversal_definition': 'no N has negative paired hit lift in 2 or more of 2022/2023/2024',
 'verdict': 'NO_PROMOTION'}

In [2]:
diagnostics=pd.read_csv(OUT/'model_diagnostics.csv')
diagnostics

,family,rows,issues,issue_weighted_log_loss,issue_weighted_brier,mean_prediction,actual_rate
0,S2_LOGISTIC,47788,157,0.569284,0.190784,0.745245,0.742159
1,S3_HGB,47788,157,0.569543,0.190881,0.743389,0.742159
2,S1_CALIBRATED_MASS,47788,157,0.569559,0.190824,0.744410,0.742159
3,S0_RAW_MASS,47788,157,0.569599,0.190797,0.739771,0.742159


In [3]:
summary=pd.read_csv(OUT/'selection_summary.csv', dtype={'period':str})
summary

,arm,n,period,issues,match_top2_hit_rate,issue_hit_rate,mean_predicted_joint_hit
0,S0_RAW_MASS,3,2019,53,0.842767,0.660377,0.644662
1,S0_RAW_MASS,3,2020,30,0.722222,0.333333,0.574406
2,S0_RAW_MASS,3,2021,48,0.770833,0.520833,0.569368
3,S0_RAW_MASS,3,2022,52,0.794872,0.576923,0.539601
4,S0_RAW_MASS,3,2023,48,0.826389,0.562500,0.544514
5,S0_RAW_MASS,3,2024,54,0.870370,0.666667,0.541284
6,S0_RAW_MASS,3,ALL_2019_2021,131,0.788804,0.534351,0.600984
7,S0_RAW_MASS,3,ALL_2022_2024,154,0.831169,0.603896,0.541722
8,S0_RAW_MASS,4,2019,53,0.830189,0.509434,0.536961
9,S0_RAW_MASS,4,2020,29,0.732759,0.241379,0.460000


In [4]:
performance=pd.read_csv(OUT/'performance.csv', dtype={'period':str})
performance[performance.period.isin(['2022','2023','2024','ALL_2022_2024'])]

,arm,n,period,issues,stake_cny,gross_cny,pnl_cny,ending_capital_cny,capital_return,stake_roi,hit_rate,largest_payout_share,payout_hhi
3,S0_RAW_MASS,3,2022,52,53248,43391.623438,-9856.376562,90143.623438,-0.098564,-0.185103,0.576923,0.105738,0.049848
4,S0_RAW_MASS,3,2023,48,49152,41018.291724,-8133.708276,91866.291724,-0.081337,-0.165481,0.562500,0.162297,0.060633
5,S0_RAW_MASS,3,2024,54,55296,53964.049108,-1331.950892,98668.049108,-0.013320,-0.024088,0.666667,0.140393,0.046796
7,S0_RAW_MASS,3,ALL_2022_2024,154,157696,138373.964271,-19322.035729,80677.964271,-0.193220,-0.122527,0.603896,0.054751,0.017347
11,S0_RAW_MASS,4,2022,51,52224,44927.602856,-7296.397144,92703.602856,-0.072964,-0.139713,0.470588,0.115544,0.062301
12,S0_RAW_MASS,4,2023,48,49152,35656.231863,-13495.768137,86504.231863,-0.134958,-0.274572,0.437500,0.132959,0.064775
13,S0_RAW_MASS,4,2024,54,55296,61338.676921,6042.676921,106042.676921,0.060427,0.109279,0.574074,0.113870,0.050684
15,S0_RAW_MASS,4,ALL_2022_2024,153,156672,141922.511640,-14749.488360,85250.511640,-0.147495,-0.094142,0.496732,0.049214,0.019800
19,S0_RAW_MASS,5,2022,51,52224,55103.399802,2879.399802,102879.399802,0.028794,0.055136,0.392157,0.171131,0.086770
20,S0_RAW_MASS,5,2023,48,49152,30277.777341,-18874.222659,81125.777341,-0.188742,-0.383997,0.333333,0.239080,0.105691


In [5]:
risk=pd.read_csv(OUT/'risk_diagnostics.csv', dtype={'period':str})
risk[risk.period.eq('ALL_2022_2024')]

,arm,n,period,max_drawdown_cny,max_drawdown_fraction,longest_losing_run,recovery_issues
7,S0_RAW_MASS,3,ALL_2022_2024,28692.504599,0.283240,15,143
15,S0_RAW_MASS,4,ALL_2022_2024,26627.871818,0.265036,8,142
23,S0_RAW_MASS,5,ALL_2022_2024,29214.456581,0.265262,8,125
31,SELECTED_MODEL,3,ALL_2022_2024,33407.573735,0.326978,13,155
39,SELECTED_MODEL,4,ALL_2022_2024,31785.873568,0.297026,13,146
47,SELECTED_MODEL,5,ALL_2022_2024,41993.491934,0.399326,16,151


In [6]:
contrasts=pd.read_csv(OUT/'matched_contrasts.csv', dtype={'period':str})
contrasts[contrasts.period.isin(['2022','2023','2024','ALL_2022_2024'])]

,n,period,common_issues,s0_pnl_cny,candidate_pnl_cny,s0_stake_roi,candidate_stake_roi,pnl_lift_cny,mean_pnl_lift_cny,hit_rate_lift,...,largest_abs_paired_delta_cny,s0_max_drawdown_fraction,candidate_max_drawdown_fraction,s0_longest_losing_run,candidate_longest_losing_run,s0_recovery_issues,candidate_recovery_issues,bootstrap_mean_lift_p05_cny,bootstrap_mean_lift_p50_cny,bootstrap_mean_lift_p95_cny
3,3,2022,52,-9856.376562,-8531.894615,-0.185103,-0.160229,1324.481948,25.470807,-0.019231,...,2978.463737,0.110142,0.114020,5,12,41,47,-141.933770,19.369291,205.731206
4,3,2023,47,-8767.222600,-10375.359430,-0.182165,-0.215578,-1608.136829,-34.215677,0.063830,...,4082.217109,0.123439,0.118665,17,12,45,46,-244.632081,-31.660906,156.798059
5,3,2024,54,-1331.950892,-1500.195683,-0.024088,-0.027130,-168.244791,-3.115644,0.018519,...,6561.038877,0.100543,0.105308,8,8,51,49,-313.982705,8.560984,273.304882
7,3,ALL_2022_2024,153,-19955.550054,-20407.449727,-0.127372,-0.130256,-451.899673,-2.953593,0.019608,...,6561.038877,0.289494,0.295292,17,12,142,148,-142.277194,-2.863680,128.503579
11,4,2022,51,-7296.397144,-3326.855390,-0.139713,-0.063704,3969.541754,77.834152,-0.019608,...,5562.329739,0.080149,0.107715,5,13,40,41,-219.110551,77.622252,400.907595
12,4,2023,46,-14346.071197,-8324.574759,-0.304562,-0.176728,6021.496438,130.902096,0.065217,...,4347.747897,0.195009,0.138946,8,7,45,44,-101.749144,120.455760,393.060893
13,4,2024,54,6042.676921,-1639.980045,0.109279,-0.029658,-7682.656966,-142.271425,0.000000,...,6984.637984,0.092810,0.122683,5,8,32,49,-500.011888,-137.552674,198.322285
15,4,ALL_2022_2024,151,-15599.791420,-13291.410194,-0.100889,-0.085960,2308.381226,15.287293,0.013245,...,6984.637984,0.273499,0.276875,8,13,140,141,-164.790483,14.534639,194.780343
19,5,2022,50,3903.399802,-4481.864292,0.076238,-0.087536,-8385.264094,-167.705282,-0.080000,...,6547.848221,0.081466,0.137206,7,16,22,46,-617.989677,-170.781734,288.248279
20,5,2023,46,-19710.376270,-13880.339657,-0.418444,-0.294674,5830.036613,126.739926,0.021739,...,4831.243550,0.249333,0.165754,7,14,45,44,-207.381229,124.068873,473.435724
